# Loading & Visualising Financial Data — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**

In [ ]:
!pip install yfinance pandas-datareader seaborn --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# FHNW corporate palette
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'

# Global plotting defaults — apply once, every chart benefits
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.25, 'font.size':11,
})
print('✓ Libraries loaded.')

---
# Solution 1 — Match the Data Source

| # | Task | Library |
|---|------|---------|
| a | Tesla daily prices | **yfinance** |
| b | US 10-year Treasury yield | **pandas-datareader (FRED)** |
| c | EUR/USD vs CHF/USD daily | **yfinance** (`EURUSD=X`, `CHFUSD=X`) or **FRED** (`DEXUSEU`, `DEXSZUS`) |
| d | Switzerland GDP per capita | **wbgapi** (World Bank) |
| e | Bitcoin daily prices | **yfinance** (`BTC-USD`) |
| f | US CPI monthly | **pandas-datareader (FRED)** (`CPIAUCSL`) |

---
# Solution 2 — Download & Inspect a Swiss Stock

In [ ]:
TICKER = 'NESN.SW'
df_n = yf.download(TICKER, start='2019-01-01', end='2024-12-31',
                    auto_adjust=True, progress=False)

print(f'Stock: {TICKER}')
print(f'Shape: {df_n.shape}')
print(f'Range: {df_n.index[0].date()} → {df_n.index[-1].date()}')
print(f'\nMissing values per column:')
print(df_n.isnull().sum())

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df_n.index, df_n['Close'], color='black', lw=1.4)
ax.set_title(f'{TICKER} — Adjusted Close', loc='left',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Price (CHF)')
plt.tight_layout(); plt.show()

**Answers:**
1. NESN.SW has roughly 1500 trading days over the 6-year window — close to ~250 per year, slightly fewer than US (which has ~252) because of different Swiss holidays (e.g. Berchtoldstag, Auffahrt, Bundesfeier).
2. Typically zero or a handful of NaNs from data-vendor gaps. yfinance is reliable for blue-chip Swiss stocks.
3. CHF (Swiss Francs). yfinance always reports prices in the listing-exchange currency. The `.SW` suffix indicates SIX Swiss Exchange.

---
# Solution 3 — Macro Data from FRED

In [ ]:
series = ['DGS10', 'DGS2', 'MORTGAGE30US']
data = {s: web.DataReader(s, 'fred', '2010-01-01', '2024-12-31') for s in series}

rates = pd.concat([d[s] for s, d in data.items()], axis=1).dropna()
rates.columns = series

fig, ax = plt.subplots(figsize=(12, 5))
for s, c in zip(series, ['black', ORANGE, RED]):
    ax.plot(rates.index, rates[s], color=c, lw=1.3, label=s)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=3, frameon=False, fontsize=11)
ax.set_title('US Interest Rates 2010–2024 (FRED)', loc='left',
             fontweight='bold', fontsize=13, pad=28)
ax.set_ylabel('Rate (%)')
plt.tight_layout(); plt.show()

spread = rates['MORTGAGE30US'] - rates['DGS10']
print(f'Mortgage-Treasury spread — mean: {spread.mean():.2f}pp, std: {spread.std():.2f}pp')

**Answers:**
1. The 30-year mortgage rate typically sits 1.5–2.5 percentage points above the 10-year Treasury — a credit/duration premium for prepayment and default risk.
2. The 2-year exceeds the 10-year (yield curve inversion) in 2019 briefly and especially during 2022–2023. Markets read inversion as a recession warning because it signals expected near-term Fed cuts (i.e. an anticipated economic slowdown).
3. They share the same underlying drivers — Federal Reserve policy, inflation expectations, and economic growth — so they move broadly together, with rate-sensitive products (mortgages) lagging slightly behind market rates.

---
# Solution 4 — Cleaning a Multi-Asset Panel

In [ ]:
tickers4 = ['AAPL', 'NESN.SW', 'BTC-USD', 'EURUSD=X']
raw4 = yf.download(tickers4, start='2023-01-01', end='2024-01-01',
                    auto_adjust=True, progress=False)
p4 = raw4['Close']

print('Per-ticker observations and NaN count for 2023:')
for col in p4.columns:
    obs = p4[col].dropna().shape[0]
    nan = p4[col].isnull().sum()
    print(f'  {col:<12}  obs={obs:>3}   NaN={nan:>3}')

drop  = p4.dropna()
ffill = p4.ffill().dropna()
print(f'\nAfter dropna():        {len(drop)} rows  (strict intersection)')
print(f'After ffill().dropna(): {len(ffill)} rows  (forward-fill then drop)')

**Answers:**
1. BTC-USD has the most rows because crypto trades 365 days a year (no weekends, no holidays). Equities and FX trade ~252 days.
2. After strict `dropna()` you keep only days where every market traded — typically ~250 rows. The bottleneck is the equities (US/Swiss), not BTC.
3. Forward-fill is appropriate for *price levels* on weekends or holidays — the price didn't disappear, it just wasn't quoted. Use it for portfolio NAV computations. **Never** forward-fill returns: it injects fake zeros.

---
# Solution 5 — Style Transformation

In [ ]:
tk = ['AAPL', 'MSFT']
px = yf.download(tk, start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close'].dropna()

# --- Default (ugly) ---
with plt.rc_context({'axes.spines.top':True, 'axes.spines.right':True,
                      'axes.grid':False}):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(px.index, px['AAPL'], label='AAPL')
    ax.plot(px.index, px['MSFT'], label='MSFT')
    ax.legend(loc='upper left')
    ax.set_title('AAPL vs MSFT')
    plt.tight_layout(); plt.show()

In [ ]:
# --- Publication quality (5 tricks) ---
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(px.index, px['AAPL'], color='black', lw=1.6, label='AAPL')
ax.plot(px.index, px['MSFT'], color=ORANGE, lw=1.6, label='MSFT')

covid_low = px['AAPL'][(px.index>='2020-02-20')&(px.index<='2020-04-30')].idxmin()
ax.annotate('COVID crash',
            xy=(covid_low, px['AAPL'].loc[covid_low]),
            xytext=(pd.Timestamp('2019-04-01'), px['AAPL'].loc[covid_low] - 30),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2),
            fontsize=11, color=RED, fontweight='bold')

ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=2, frameon=False, fontsize=11)
ax.set_title('Apple vs. Microsoft — Adjusted Close, 2019–2024',
             loc='left', fontweight='bold', fontsize=13, pad=28)
ax.set_ylabel('Price (USD)')
plt.tight_layout(); plt.show()

**Answers:**
1. In the default plot, the legend at `upper left` overlaps the price series early in the sample (when prices were lowest) — exactly when you'd want to read the chart most carefully.
2. The COVID crash in March 2020 is the most informative event to annotate — it's the largest drawdown in the sample and provides instant temporal context for any reader.
3. For *this* chart, **Trick 5 (legend outside)** is the single biggest readability win — but **Trick 3 (clear left-aligned title with units)** transforms the chart from 'data dump' to 'communication'.

---
# Solution 6 — Multi-Asset Comparison with COVID Annotation

In [ ]:
tk = ['SPY', 'TLT', 'GLD', 'BTC-USD']
px = yf.download(tk, start='2019-06-01', end='2021-06-01',
                 auto_adjust=True, progress=False)['Close'].dropna()
cum = px.div(px.iloc[0]).sub(1) * 100   # cumulative return in %, all start at 0

fig, ax = plt.subplots(figsize=(12, 5.5))
colors = {'SPY':'black', 'TLT':BLUE, 'GLD':ORANGE, 'BTC-USD':RED}
for col in cum.columns:
    ax.plot(cum.index, cum[col], color=colors[col], lw=1.6, label=col)

ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-30'),
           color=YELLOW, alpha=0.25, label='COVID crash')
ax.axhline(0, color=GREY, lw=0.8)

spy_worst = px['SPY'].pct_change().idxmin()
ax.annotate(f'SPY worst day:\n{spy_worst.date()}',
            xy=(spy_worst, cum['SPY'].loc[spy_worst]),
            xytext=(spy_worst + pd.Timedelta(days=120),
                    cum['SPY'].loc[spy_worst] - 25),
            arrowprops=dict(arrowstyle='->', color=RED, lw=1.2),
            fontsize=10, color=RED, fontweight='bold')

ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=5, frameon=False, fontsize=11)
ax.set_title('Asset Class Performance Through COVID, June 2019 – June 2021',
             loc='left', fontweight='bold', fontsize=13, pad=28)
ax.set_ylabel('Cumulative return (%)')
plt.tight_layout(); plt.show()

covid = cum.loc['2020-02-20':'2020-04-30']
max_dd = (covid - covid.cummax()).min().sort_values()
print('Peak-to-trough drawdown during COVID window:')
for k, v in max_dd.items():
    print(f'  {k:<10}  {v:.1f}%')

# Recovery time for SPY
pre_high = px['SPY'].loc[:'2020-02-19'].max()
post = px['SPY'].loc['2020-03-23':]
recover = post[post >= pre_high].index[0]
print(f'\nSPY pre-crash high: {px["SPY"].loc[:"2020-02-19"].idxmax().date()}')
print(f'SPY recovered to that level on: {recover.date()}')

**Answers:**
1. SPY had the largest equity drawdown in the COVID window (~−34%); BTC-USD also fell sharply (~−40% intra-window).
2. TLT (long Treasuries) and GLD (gold) acted as safe havens — both held up or gained, demonstrating the textbook negative correlation of duration assets to equity stress.
3. SPY recovered to its pre-crash high in roughly 5 months — by mid-August 2020 — far faster than typical bear-market recoveries, thanks to massive Fed and fiscal stimulus.

---
# Solution 7 — Build Your Own load_panel

In [ ]:
def load_panel_v2(tickers, start, end, return_type='simple'):
    """Returns (daily_returns, cumulative_returns) tuple."""
    if return_type not in ('simple', 'log'):
        raise ValueError(f"return_type must be 'simple' or 'log', got {return_type!r}")
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].dropna(how='all')
    if return_type == 'simple':
        ret = prices.pct_change().dropna()
        cum = (1 + ret).cumprod() - 1
    else:
        ret = np.log(prices / prices.shift(1)).dropna()
        cum = ret.cumsum()
    return ret, cum

ret_s, cum_s = load_panel_v2(['AAPL','MSFT'], '2023-01-01', '2024-12-31', 'simple')
ret_l, cum_l = load_panel_v2(['AAPL','MSFT'], '2023-01-01', '2024-12-31', 'log')

print('Final cumulative — simple:')
print((cum_s.iloc[-1] * 100).round(2))
print('\nFinal cumulative — log:')
print((cum_l.iloc[-1] * 100).round(2))

**Answers:**
1. Simple cumulative returns are slightly *higher* than log cumulative returns over the same window — for a stock that gains ~50%, the gap is a few percentage points. The two converge for short windows or small daily moves.
2. Log returns are time-additive — they sum across periods. Statistical models (regressions, GARCH, factor models) almost always work in log returns. Simple returns are easier for client reporting.
3. The validation step catches typos like `'logarithmic'` or `'pct'` immediately rather than producing silently wrong output. Always validate the small set of allowed inputs.

---
# Solution 8 — Save & Reload

In [ ]:
import time

t0 = time.time()
px = yf.download('AAPL', start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
t_download = time.time() - t0
px.to_csv('aapl_prices.csv', header=['Close'])

t0 = time.time()
px_loaded = pd.read_csv('aapl_prices.csv', index_col=0, parse_dates=True)['Close']
t_reload = time.time() - t0

print(f'Download time: {t_download:.2f} s')
print(f'Reload time:   {t_reload:.4f} s')
print(f'Speed-up:      {t_download / max(t_reload, 1e-6):.0f}×')
print(f'\nShapes match? {px.shape == px_loaded.shape}')
print(f'Dtypes match? {px.dtype == px_loaded.dtype}')

**Answers:**
1. Reloading from CSV is typically 100–500× faster than downloading — milliseconds vs. seconds. The exact ratio depends on network speed.
2. `.parquet` (via `pyarrow`) is faster and preserves dtypes precisely; `.feather` is similar. CSV is human-readable and universally supported, which is its main advantage for teaching.
3. Without `parse_dates=True`, pandas reads the date column as plain strings — slicing by date and resampling won't work. Always parse dates explicitly when reloading time-series CSVs.

---
# Solution 9 — Currency Mix

In [ ]:
px = yf.download(['AAPL','NESN.SW'], start='2023-01-01', end='2024-01-01',
                  auto_adjust=True, progress=False)['Close'].dropna()

# Bad: prices on the same axis (different currencies, very different magnitudes)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(px.index, px['AAPL'],    color='black', lw=1.4, label='AAPL (USD)')
ax.plot(px.index, px['NESN.SW'], color=ORANGE, lw=1.4, label='NESN.SW (CHF)')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
ax.set_title('Bad: Mixed-Currency Prices on One Axis', loc='left',
             fontweight='bold', pad=28)
plt.tight_layout(); plt.show()
print('→ Y-axis values mix USD and CHF — magnitudes are not comparable.\n')

# Good: cumulative returns from the start of the period
cum = px.div(px.iloc[0]).sub(1) * 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cum.index, cum['AAPL'],    color='black', lw=1.4, label='AAPL')
ax.plot(cum.index, cum['NESN.SW'], color=ORANGE, lw=1.4, label='NESN.SW')
ax.axhline(0, color=GREY, lw=0.8)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
ax.set_title('Good: Cumulative Returns — Currency-Free Comparison',
             loc='left', fontweight='bold', pad=28)
ax.set_ylabel('Cumulative return (%)')
plt.tight_layout(); plt.show()

**Answers:**
1. A USD price and a CHF price are quoted in different units — like comparing 5 metres to 5 kilograms. The numerical value carries no comparable meaning across currencies.
2. *Returns* are dimensionless ratios (or differences in log-prices). The unit cancels out in the percent change calculation, making them directly comparable across currencies and asset classes.
3. Convert prices to a common currency when you need an *FX-aware* analysis — e.g. the total-return performance for a USD investor holding a Swiss stock (which depends on both the stock and the CHF/USD move). For pure performance comparison, returns are sufficient and simpler.

---
# Solution 10 — Mini-Dashboard (theme: Swiss blue chips)

In [ ]:
def load_panel(tickers, start, end):
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].dropna(how='all')
    return prices.pct_change().dropna()

swiss = ['NESN.SW', 'NOVN.SW', 'ROG.SW', 'UBSG.SW']
ret = load_panel(swiss, '2020-01-01', '2024-12-31')
cum = (1 + ret).cumprod() - 1

fig, ax = plt.subplots(figsize=(11, 5))
colors_s = ['black', ORANGE, BLUE, RED]
for col, c in zip(cum.columns, colors_s):
    ax.plot(cum.index, cum[col]*100, color=c, lw=1.6, label=col)
ax.axhline(0, color=GREY, lw=0.8)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=4, frameon=False)
ax.set_title('Swiss Blue Chips — Cumulative Returns 2020–2024',
             loc='left', fontweight='bold', pad=28)
ax.set_ylabel('Cumulative return (%)')
plt.tight_layout(); plt.show()

summary = pd.DataFrame({
    'Total Return %':   (cum.iloc[-1]*100).round(2),
    'Annual Vol %':     (ret.std()*np.sqrt(252)*100).round(2),
    'Worst Day %':      (ret.min()*100).round(2),
    'Best Day %':       (ret.max()*100).round(2),
})
print(summary)

**Narrative:**
Swiss blue chips diverged sharply over 2020–2024. Pharma and consumer staples (Nestlé, Roche, Novartis) delivered moderate, low-volatility performance consistent with their defensive profile. UBS, by contrast, had the highest volatility and the wildest path — driven first by COVID, then by the Credit Suisse rescue acquisition in 2023 — but ended the period with the strongest total return. The dispersion in volatility (roughly 15% for staples vs 30%+ for the bank) shows clearly why blanket 'Swiss equities' is a misleading category for portfolio construction.

---
# 🔥 Challenge — yfinance + FRED Together

In [ ]:
spx = yf.download('^GSPC', start='2010-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close'].squeeze()
y10 = web.DataReader('DGS10', 'fred', '2010-01-01', '2024-12-31')['DGS10']

common = spx.index.intersection(y10.index)
spx, y10 = spx.loc[common], y10.loc[common]

fig, ax_l = plt.subplots(figsize=(12, 5))
ax_r = ax_l.twinx()
ax_r.spines['top'].set_visible(False)   # match the global no-top-spine style

l1, = ax_l.plot(spx.index, spx.values, color='black', lw=1.4, label='S&P 500')
l2, = ax_r.plot(y10.index, y10.values, color=RED,     lw=1.2, label='10y Treasury yield')
ax_l.set_ylabel('S&P 500 level',   color='black')
ax_r.set_ylabel('10y yield (%)',   color=RED)
ax_r.grid(False)   # avoid double-grid clutter on twin axis

# Single combined legend — outside the plot, no overlap with data
ax_l.legend([l1, l2], ['S&P 500', '10y Treasury yield'],
            loc='lower center', bbox_to_anchor=(0.5, 1.02),
            ncol=2, frameon=False, fontsize=11)

ax_l.set_title('S&P 500 vs. 10-Year Treasury Yield, 2010–2024',
               loc='left', fontweight='bold', fontsize=13, pad=28)
plt.tight_layout(); plt.show()

# Correlation of returns vs. yield changes
spx_ret  = spx.pct_change().dropna()
y10_diff = y10.diff().dropna()
common2 = spx_ret.index.intersection(y10_diff.index)
corr = spx_ret.loc[common2].corr(y10_diff.loc[common2])
print(f'\nCorrelation, daily SPX returns vs. daily 10y yield changes: {corr:.3f}')
print('(Typically slightly positive — equities and yields tend to move together when growth expectations shift.)')

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*